# Analisando as Transformadas em Tempo-Frequência

Este notebook tem papel importante no projeto, pois é nele que os sinais de áudio respiratório, originalmente representações unidimensionais (1D), são transformados em representações bidimensionais (2D) na forma de espectrogramas, que servirão de entrada para o modelo de classificação por imagem. A escolha de qual transformação utilizar tem impacto direto na qualidade da informação fornecida ao modelo, motivo pelo qual diferentes representações são exploradas e comparadas visualmente antes da definição do conjunto final de features.

Todas as transformações analisadas derivam da Short-Time Fourier Transform (STFT), que divide o sinal de áudio em janelas curtas e calcula a Transformada de Fourier em cada uma delas:

$$X(m,k)=\sum_{n=0}^{N-1} x[n+mH] \, w[n] \, e^{-j2\pi kn/N}$$

Onde $x[n]$ é o sinal de áudio, $w[n]$ é a janela aplicada (geralmente Hann), $N$ é o tamanho da FFT, $H$ é o hop length, $m$ é o índice temporal e $k$ é o bin de frequência. A partir do coeficiente complexo $X(m,k)$, são derivadas as transformações analisadas neste notebook.

### Parte Real e Parte Imaginária da STFT (RealSTFT / ImagSTFT)

Correspondem, respectivamente, à parte real e à parte imaginária do coeficiente complexo da STFT:

$$RealSTFT(m,k) = \Re(X(m,k))$$
$$ImagSTFT(m,k) = \Im(X(m,k))$$

Isoladamente, essas componentes não representam energia de forma direta e tendem a oscilar simetricamente em torno de zero para sinais de áudio típicos.

### Magnitude do Espectro (MagSTFT)

Representa a energia ou amplitude de cada frequência ao longo do tempo, calculada a partir das partes real e imaginária do coeficiente complexo:

$$|X(m,k)| = \sqrt{\Re(X)^2 + \Im(X)^2}$$

É a representação mais diretamente interpretável em termos de energia do sinal.

### Fase Espectral (Phase)

Corresponde ao ângulo de fase do coeficiente complexo:

$$\phi(m,k) = \tan^{-1}\left(\frac{\Im(X(m,k))}{\Re(X(m,k))}\right)$$

Por natureza, a fase tende a apresentar comportamento próximo ao ruído aleatório, com baixo valor informativo direto na maioria das aplicações de classificação por áudio.

### Mel Spectrogram

Calculado em duas etapas. Primeiro, obtém-se o espectrograma de potência:

$$P(m,k) = |X(m,k)|^2$$

Em seguida, aplica-se um banco de filtros Mel:

$$M(m,r) = \sum_{k} H_r(k) \, P(m,k)$$

Onde $H_r(k)$ é o filtro triangular Mel e $r$ é o índice do filtro. Essa transformação comprime as frequências mais altas, de modo a aproximar a percepção auditiva humana, que é menos sensível a variações em altas frequências do que em baixas frequências.

### Mel Frequency Cepstral Coefficients (MFCC)

Aplica-se a Discrete Cosine Transform (DCT) sobre o logaritmo do Mel Spectrogram:

$$L(m,r) = \log(M(m,r))$$

$$C(m,n) = \sum_{r=0}^{R-1} L(m,r) \cos\left[\frac{\pi n}{R}\left(r+\frac{1}{2}\right)\right]$$

Onde $C(m,n)$ é o coeficiente MFCC e $R$ é o número de filtros Mel. Essa transformação decorrelaciona a informação espectral, concentrando a maior parte da energia nos primeiros coeficientes.

### Delta MFCC (MFCCDelta)

Corresponde à derivada temporal dos coeficientes MFCC, calculada por meio de uma regressão linear local sobre janelas vizinhas no tempo:

$$\Delta c_t = \frac{\sum_{n=1}^{N} n(c_{t+n} - c_{t-n})}{2\sum_{n=1}^{N} n^2}$$

Captura a taxa de variação do espectro ao longo do tempo, descrevendo a dinâmica do som respiratório, e não apenas seu conteúdo espectral estático.

### Chroma

Agrega a energia espectral em classes de pitch (correspondentes às notas da escala cromática: C, C#, D, ...):

$$Chroma(p,t) = \sum_{k \in K_p} |X(t,k)|$$

Onde $p$ é a classe cromática e $K_p$ são os bins de frequência associados àquela nota. Trata-se de uma representação originalmente concebida para conteúdo musical tonal, de aplicabilidade conceitualmente limitada a sons respiratórios.

---

Com base nessas transformações, este notebook está organizado da seguinte forma: inicialmente, é definido um subconjunto de classes de diagnóstico com representatividade suficiente no dataset combinado, seguido pela definição de funções auxiliares para amostragem e visualização. Em seguida, o áudio é padronizado em taxa de amostragem e duração fixas, e as oito transformações descritas acima são aplicadas a uma amostra de cada classe selecionada, permitindo a comparação visual entre os padrões espectrais associados a cada diagnóstico. Durante essa análise, outras representações, como o Spectral Contrast e a Constant-Q Transform (CQT), também foram investigadas, mas não compuseram o conjunto final adotado para a extração de features e o treinamento dos modelos, etapa realizada posteriormente no notebook `5_preprocess_features.ipynb`.

## Configuração do Ambiente e Definição de Constantes
O bloco importa as bibliotecas e módulos necessários para a análise. Além de librosa e matplotlib, são importadas as classes LungSoundAudio e LungSoundFeatures do módulo modules.lungsound, que representam, respectivamente, o áudio bruto e as features extraídas após uma transformação. São importadas também CombinedAudioDataset e AudioDataset do módulo de datasets, além de todas as transformações definidas em modules.transforms. É definida uma seed fixa (SEED = 42) para garantir a reprodutibilidade da seleção aleatória de amostras ao longo do notebook.
A lista SELECTED_CLASSES restringe a análise a oito das doze classes originais (Asthma, Bronchiectasis, Bronchiolitis, COPD, Healthy, Lung Fibrosis, Pneumonia e URTI). As classes Bronchitis, Heart Failure, LRTI e Pleural Effusion são comentadas e, portanto, excluídas da análise, conforme indicado pelo próprio comentário no código, que justifica essa exclusão pelo baixo número de amostras disponíveis para essas classes no dataset combinado.

In [ ]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

# Set random seed for reproducibility
SEED = 42
import random
random.seed(SEED)

import librosa
from matplotlib import pyplot as plt

from modules.lungsound import LungSoundAudio, LungSoundFeatures
from modules.datasets import CombinedAudioDataset, AudioDataset
from modules.transforms import *

# NOTE: Some classes have very few samples, so we will focus on a subset of classes for the analysis.
SELECTED_CLASSES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    # "Bronchitis",
    "COPD",
    "Healthy",
    # "Heart Failure",
    # "LRTI",
    "Lung Fibrosis",
    # "Pleural Effusion",
    "Pneumonia",
    "URTI",
]

O caminho DATA_FOLDER é definido apontando para a pasta de dados brutos, com uma verificação simples para alertar caso o diretório não exista.

In [ ]:
DATA_FOLDER = os.path.join(os.path.dirname(os.getcwd()), "data/raw")
if not os.path.exists(DATA_FOLDER):
    print(f"Data directory not found at {DATA_FOLDER}. Please ensure the data is downloaded and placed in the correct location.")

A função get_samples_by_label recebe um dataset e retorna um dicionário que mapeia cada classe de diagnóstico a um índice de amostra selecionado aleatoriamente. Para cada rótulo único presente na coluna Diagnosis, é filtrado o subconjunto de amostras correspondente e escolhido um índice por meio de random.choice, garantindo que exatamente uma amostra de cada classe seja selecionada para a visualização.
A função plot_features_by_label recebe o dataset e, opcionalmente, o dicionário de amostras gerado pela função anterior. Caso nenhum dicionário seja fornecido, a própria função chama get_samples_by_label internamente e imprime os índices sorteados para cada classe. São definidos oito extratores de features, todos parametrizados com n_fft=2048 e hop_length=512: STFT magnitude (MagSTFT), STFT parte imaginária (ImagSTFT), STFT parte real (RealSTFT), fase (Phase), espectrograma Mel (MelSpectrogram, com 128 mels), MFCC (MFCC, com 128 coeficientes), MFCC com derivada (MFCCDelta) e Chroma (Chroma, com 128 bins).
Para cada extrator, a função monta uma grade de subplots, com no máximo quatro colunas, e aplica a transformação correspondente à amostra de cada classe sorteada. O resultado da transformação, um objeto LungSoundFeatures, é exibido por meio de librosa.display.specshow, com uma barra de cores (colorbar) ao lado de cada imagem. O formato (shape) da matriz de features resultante é impresso a cada iteração, e o título geral da figura é definido com o nome do extrator utilizado.

In [ ]:
def get_samples_by_label(dataset: AudioDataset) -> dict:
    """
    Get one sample for each label in the dataset.
    Args:
        dataset (AudioDataset): The dataset to sample from.
    Returns:
        dict: A dictionary mapping each label to a random sample index.
    """
    data = dataset.data
    samples = {}
    for label in sorted(data["Diagnosis"].unique()):
        label_samples = data[data["Diagnosis"] == label]
        random_idx = random.choice(label_samples.index.tolist())
        samples[label] = random_idx
    return samples

def plot_features_by_label(dataset: AudioDataset, samples: dict | None = None) -> None:
    """
    Plot features for one sample of each label in the dataset.
    Args:
        dataset (AudioDataset): The dataset to sample from.
        samples (dict, optional): A dictionary mapping each label to a sample index. If None, samples will be randomly selected.
    """
    if samples is None:
        samples = get_samples_by_label(dataset)
        print("Sampled records:")
        print("----------------")
        for label, sample in samples.items():
            print(f"Class: {label}, idx: {sample}")
        print("----------------")

    feature_extractors = [
        MagSTFT(n_fft=2048, hop_length=512),
        ImagSTFT(n_fft=2048, hop_length=512),
        RealSTFT(n_fft=2048, hop_length=512),
        Phase(n_fft=2048, hop_length=512),
        MelSpectrogram(n_fft=2048, hop_length=512, n_mels=128),
        MFCC(n_fft=2048, hop_length=512, n_mfcc=128),
        MFCCDelta(n_fft=2048, hop_length=512, n_mfcc=128),
        Chroma(n_fft=2048, hop_length=512, n_chroma=128),
    ]

    for extractor in feature_extractors:
        extractor_name = extractor.name
        num_classes = len(samples)
        num_rows = (num_classes + 3) // 4
        num_cols = min(num_classes, 4)
        figure, axes = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 3 * num_rows), constrained_layout=True)
        axes = axes.flatten()

        params = extractor.plot_params

        for i, (class_name, idx) in enumerate(samples.items()):
            sample, _ = dataset[idx]
            sample_transformed = extractor(sample)
            sample: LungSoundAudio
            sample_transformed: LungSoundFeatures

            ax = axes[i]
            img = librosa.display.specshow(sample_transformed.features, sr=sample_transformed.sr, ax=ax, **params)
            ax.set_title(class_name)
            plt.colorbar(img, ax=ax)
            print(f"Sample shape for {extractor_name}: {sample_transformed.features.shape}")

        figure.suptitle(f"{extractor_name}")
        plt.show()

São definidos os parâmetros de padronização do áudio: taxa de amostragem alvo de 22050 Hz (TARGET_SR) e duração alvo de 5 segundos (TARGET_DURATION). É construído um pipeline de transformações por meio da classe Compose, que encadeia três etapas: Resample, responsável por reamostrar o áudio para a taxa definida; PadOrTrim, que ajusta a duração da gravação para o valor alvo, truncando ou preenchendo conforme necessário; e NormalizeAudio, que normaliza a amplitude do sinal.
Em seguida, é instanciado o dataset combinado por meio de CombinedAudioDataset, agora restrito às classes definidas em SELECTED_CLASSES, com o pipeline de transformações aplicado a cada amostra e com a seed fixa definida anteriormente, garantindo reprodutibilidade na seleção aleatória das amostras. Por fim, a função plot_features_by_label é chamada sobre esse dataset, gerando as visualizações das oito transformações tempo-frequência para uma amostra de cada classe selecionada.

In [ ]:
TARGET_SR = 22050       # Hz
TARGET_DURATION = 5.0   # seconds

transforms = Compose([
    Resample(target_sr=TARGET_SR),
    PadOrTrim(target_duration=TARGET_DURATION),
    NormalizeAudio(),
])

dataset = CombinedAudioDataset(DATA_FOLDER, "all", classes=SELECTED_CLASSES, transform=transforms, random_seed=SEED)
plot_features_by_label(dataset)

In [ ]:
def plot_all_features_for_one_sample(dataset: AudioDataset, sample_idx: int) -> None:
    """
    Plot features for one sample of each label in the dataset.
    Args:
        dataset (AudioDataset): The dataset to sample from.
        sample_idx (int): The index of the sample to plot features for.
    """
    feature_extractors = [
        MagSTFT(n_fft=2048, hop_length=512),
        ImagSTFT(n_fft=2048, hop_length=512),
        RealSTFT(n_fft=2048, hop_length=512),
        Phase(n_fft=2048, hop_length=512),
        MelSpectrogram(n_fft=2048, hop_length=512, n_mels=128),
        MFCC(n_fft=2048, hop_length=512, n_mfcc=128),
        MFCCDelta(n_fft=2048, hop_length=512, n_mfcc=128),
        Chroma(n_fft=2048, hop_length=512, n_chroma=128),
    ]

    sample, label = dataset[sample_idx]
    class_name = dataset.data.loc[sample_idx, "Diagnosis"]
    print(f"Sampled record: Class: {class_name}, idx: {sample_idx}")
    print(f"Sample shape: {sample.audio.shape}, Sample sr: {sample.sr}")

    num_extractors = len(feature_extractors)
    num_rows = (num_extractors + 3) // 4
    num_cols = min(num_extractors, 4)
    figure, axes = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 3 * num_rows), constrained_layout=True)
    axes = axes.flatten()

    for i, extractor in enumerate(feature_extractors):
        extractor_name = extractor.name
        params = extractor.plot_params
        sample_transformed = extractor(sample)
        sample: LungSoundAudio
        sample_transformed: LungSoundFeatures
        ax = axes[i]
        img = librosa.display.specshow(sample_transformed.features, sr=sample_transformed.sr, ax=ax, **params)
        ax.set_title(f"{extractor_name}")
        plt.colorbar(img, ax=ax)

    figure.suptitle(f"Class: {class_name}")
    plt.show()

plot_all_features_for_one_sample(dataset, sample_idx=3)

### MagSTFT
A análise evidencia diferenças entre os padrões respiratórios das patologias e o sinal considerado saudável. No caso do sinal Healthy, observa-se um padrão espectral periódico, com energia concentrada nas baixas frequências (abaixo de 512 Hz) e picos que se repetem ao longo do tempo, compatíveis com o ciclo inspiração-expiração. Há menor dispersão de energia para as frequências médias e altas, indicando um fluxo aéreo pulmonar mais regular. Já nos sinais patológicos, percebe-se aumento da complexidade espectral e mudanças na distribuição de energia ao longo do tempo e da frequência:

- Asthma apresenta regiões intermitentes de energia distribuídas em faixas específicas de frequência, com padrão menos periódico que o saudável, compatível com a presença de sibilos.

- Bronchiectasis mostra um padrão bastante energético e repetitivo ao longo do tempo, com estruturas verticais recorrentes concentradas predominantemente abaixo de 1024 Hz.

- Bronchiolitis apresenta padrão periódico com maior densidade de energia em frequências intermediárias (até aproximadamente 2048 Hz), em comparação ao observado no sinal saudável.

- COPD exibe energia mais difusa e fragmentada ao longo do tempo, sem ciclos respiratórios tão bem definidos, sugerindo maior irregularidade no padrão acústico.

- Pneumonia apresenta uma região isolada de alta intensidade espectral próxima de 1.8 segundos, possivelmente associada a um evento de tosse, além de maior espalhamento de energia nas frequências baixas e médias ao longo da gravação.

- URTI mostra maior irregularidade temporal e dispersão de energia em frequências médias, em comparação ao sinal saudável, embora sem padrão tão estruturado quanto o observado em Bronchiectasis ou Bronchiolitis.

No caso de Lung Fibrosis, observa-se comportamento distinto dos demais: a energia espectral concentra-se quase exclusivamente nas frequências mais baixas, com o restante da imagem apresentando intensidade consistentemente reduzida ao longo de toda a gravação. Esse padrão pode refletir uma característica real do sinal acústico desta amostra, mas também levanta a possibilidade de tratar-se de uma gravação de baixa energia geral ou com maior presença de silêncio, o que deverá ser verificado pontualmente antes de se assumir esse comportamento como representativo da classe.

### ImagSTFT
A representação ImagSTFT corresponde à parte imaginária da Transformada de Fourier de Curto Termo, mantendo o mesmo formato de matriz (1025, 216) observado no MagSTFT. Diferentemente da magnitude, essa componente não representa diretamente a energia do sinal, mas sim a parte imaginária do espectro complexo, que tende a oscilar simetricamente em torno de zero para sinais de áudio típicos, com baixa amplitude na ausência de eventos acústicos relevantes.

- Asthma apresenta coloração predominantemente neutra, com pequena variação positiva concentrada nas frequências mais baixas (abaixo de 128 Hz), associada aos eventos respiratórios já identificados no MagSTFT.

- Bronchiectasis exibe padrão semelhante, com leve estrutura periódica visível apenas nas frequências mais baixas, mantendo amplitude reduzida ao longo de praticamente toda a imagem.

- Bronchiolitis apresenta variação um pouco mais perceptível nas frequências baixas, ainda com amplitude pequena, compatível com a maior densidade de eventos respiratórios observada nesta classe.

- COPD mostra padrão majoritariamente neutro, com discreta variação nas frequências baixas, sem estrutura adicional relevante nas demais faixas espectrais.

- Healthy mantém comportamento semelhante às demais classes regulares, com pequena variação periódica nas frequências baixas, consistente com o ciclo respiratório regular já observado.

- Pneumonia e URTI apresentam padrão neutro ao longo de quase toda a imagem, com leve variação concentrada nas frequências baixas, sem distinção marcante em relação ao padrão saudável nesta representação.


No caso de Lung Fibrosis, observa-se comportamento anômalo em relação às demais classes: a imagem inteira apresenta coloração uniforme em tom avermelhado, indicando um valor de magnitude elevado e aproximadamente constante em toda a extensão de tempo e frequência, sem a variação esperada para um sinal de áudio real. Esse padrão sugere a possibilidade de um problema pontual na amostra selecionada, como um offset constante no sinal de origem ou um artefato introduzido durante o carregamento ou as etapas de pré-processamento, e não necessariamente uma característica representativa da classe. Recomenda-se inspeção específica dessa amostra (índice 960) antes de qualquer conclusão sobre o padrão acústico associado a Lung Fibrosis.

### RealSTFT
A representação RealSTFT corresponde à parte real da Transformada de Fourier de Curto Termo, mantendo o mesmo formato de matriz (1025, 216) observado nas transformações anteriores. Assim como a componente imaginária, essa representação tende a oscilar em torno de zero para sinais de áudio típicos, sem capturar diretamente a energia do sinal.

Asthma apresenta variação concentrada nas frequências baixas, com alguma dispersão ao longo do tempo, mantendo coloração predominantemente neutra nas demais faixas.
Bronchiectasis exibe padrão periódico discreto nas frequências baixas, com amplitude reduzida e coloração neutra no restante da imagem.
Bronchiolitis apresenta maior variação nas frequências baixas em comparação às classes anteriores, ainda concentrada próxima de 0 a 128 Hz.
COPD mostra padrão majoritariamente neutro, com pequena variação esparsa nas frequências mais baixas.
Healthy mantém comportamento semelhante, com variação discreta nas frequências baixas e coloração neutra no restante do espectro.
Pneumonia apresenta padrão neutro ao longo de quase toda a imagem, com leve variação concentrada nas frequências baixas.

Tanto Lung Fibrosis quanto URTI apresentam, novamente, comportamento anômalo em relação às demais classes: ambas exibem coloração uniforme em tom avermelhado em praticamente toda a extensão de tempo e frequência, sem a variação esperada para um sinal de áudio real. No caso de Lung Fibrosis, esse padrão já havia sido observado na representação ImagSTFT, o que reforça a hipótese de um problema específico nessa amostra (índice 960), possivelmente relacionado a um offset constante no sinal. Já a ocorrência desse mesmo padrão em URTI, classe que não apresentava anomalia nas transformações anteriores, sugere que se trata de um problema pontual na amostra específica de URTI (índice 108) nesta representação, e não de uma falha sistemática da transformação RealSTFT, já que as demais classes seguem comportamento esperado. Recomenda-se inspeção individual de ambas as amostras antes de se assumir que esse padrão é representativo das respectivas classes.

### Phase
A representação Phase corresponde ao ângulo de fase do espectro complexo obtido pela Transformada de Fourier de Curto Termo, mantendo o mesmo formato de matriz (1025, 216) observado nas transformações anteriores. Por natureza, a fase de um sinal de áudio tende a apresentar comportamento próximo ao ruído aleatório, distribuído entre os limites de aproximadamente -3 e 3 radianos, sem relação direta e perceptível com a energia ou a estrutura temporal do sinal.
De fato, em todas as classes observa-se um padrão visualmente semelhante a ruído, sem estruturas periódicas claras comparáveis às identificadas no MagSTFT. Um elemento comum a todas as classes é a presença de uma faixa horizontal vermelha na frequência 0 Hz, decorrente do componente DC do espectro, cuja fase é tipicamente nula ou constante por se tratar de um valor real.
Observa-se ainda, em Asthma, Healthy e Lung Fibrosis, um padrão de bandas verticais mais claras intercaladas ao ruído, sobretudo nas frequências acima de 2048 Hz, padrão não observado com a mesma intensidade nas demais classes (Bronchiectasis, Bronchiolitis, COPD, Pneumonia e URTI). Não é possível, a partir de uma única amostra por classe, determinar se esse padrão de bandas está relacionado a alguma característica específica do diagnóstico, ou se decorre de particularidades pontuais das gravações ou do equipamento utilizado.


### Mel Spetogram
A análise  evidencia diferenças entre os padrões respiratórios das patologias e o sinal considerado saudável. No caso do sinal Healthy (Saudável), observa-se uma distribuição espectral homogênea e contínua, com predominância de energia concentrada nas baixas frequências, comportamento esperado em ciclos respiratórios fisiológicos. Há menor presença de eventos impulsivos e menor variabilidade temporal, indicando estabilidade do fluxo aéreo pulmonar. Já nos sinais patológicos, percebe-se aumento da complexidade espectral e mudanças importantes na distribuição temporal da energia:

- Asthma apresenta regiões intermitentes de energia distribuídas em faixas específicas de frequência, compatíveis com a presença de sibilos.

- Bronchiectasis mostra um padrão bastante energético e repetitivo ao longo do tempo, com estruturas verticais recorrentes e ampla ocupação espectral.

- Bronchiolitis apresenta distribuição espectral mais difusa, com maior densidade de componentes em médias frequências.

- LRTI exibe eventos esparsos de energia intensa, indicando comportamento acústico menos regular que o saudável.

- Pneumonia apresenta regiões isoladas de alta intensidade espectral, além de maior espalhamento energético em frequências médias e baixas.

- URTI mostra aumento moderado da atividade espectral e maior irregularidade temporal em comparação ao saudável, embora com menor complexidade que patologias pulmonais mais severas.

No caso do COPD, observa-se um comportamento diferente dos demais. Parte do espectrograma contém atividade espectral normal do sinal respiratório, enquanto uma grande região escura aparece no restante da imagem. Isso ocorre devido ao zero padding aplicado durante o pré-processamento inicial para padronizar todos os áudios em 20 segundos — artefato que motivou a substituição dessa abordagem pela estratégia de janelamento sem padding, descrita na seção de Metodologia. Mesmo assim, na região correspondente ao áudio real, o COPD ainda apresenta um padrão mais irregular e fragmentado em comparação ao saudável, o que está de acordo com alterações respiratórias típicas da doença.

### MFCC
A representação MFCC (Mel-Frequency Cepstral Coefficients) resulta em uma matriz de formato (128, 216), correspondente aos 128 coeficientes cepstrais extraídos para cada um dos 216 quadros temporais.
Diferentemente das representações anteriores, os espectrogramas MFCC das diferentes classes apresentam aparência visual bastante semelhante entre si, predominando uma coloração avermelhada quase uniforme ao longo de toda a imagem, com uma faixa azul escura concentrada nas linhas inferiores. Esse padrão decorre de uma característica conhecida do MFCC: o primeiro coeficiente (c0), relacionado à energia logarítmica do sinal, costuma assumir valores de magnitude muito superior aos demais coeficientes. Como a escala de cores é definida com base no intervalo completo de valores da matriz, a grande diferença entre o c0 e os coeficientes de ordem mais alta comprime visualmente o contraste nas demais linhas, fazendo com que variações relevantes entre as classes fiquem pouco perceptíveis nesta visualização.
Ainda assim, é possível notar sutilmente um padrão de estrias verticais mais marcado em Asthma, Bronchiectasis e Bronchiolitis, compatível com os ciclos respiratórios já identificados nas representações anteriores, enquanto COPD, Healthy, Pneumonia e URTI aparecem com aspecto mais homogêneo. A amostra de Lung Fibrosis apresenta coloração geral mais intensa (tons de vermelho mais saturados), reforçando a observação já feita nas representações anteriores de que essa amostra específica possui características atípicas em relação às demais classes.

### MFCCDelta
A representação MFCCDelta corresponde à derivada temporal dos coeficientes MFCC, mantendo o mesmo formato de matriz (128, 216). Por se tratar de uma derivada, o valor esperado para a maior parte da imagem é próximo de zero, com variações localizadas apenas nos instantes em que há mudança mais acentuada na energia espectral, como início e fim de ciclos respiratórios.
Esse comportamento é observado em Asthma, Bronchiectasis, COPD, Healthy, Pneumonia e URTI, cujas imagens apresentam predominantemente coloração neutra (próxima de zero), com textura ruidosa de baixa amplitude distribuída ao longo do tempo e dos coeficientes, e uma faixa de maior intensidade próxima à base da imagem, possivelmente associada ao coeficiente c0.
Bronchiolitis e Lung Fibrosis, no entanto, repetem o padrão anômalo já identificado nas representações anteriores: ambas apresentam coloração uniforme em praticamente toda a extensão da imagem, em tons de laranja e azul, respectivamente, sem a variação esperada para uma derivada de sinal de áudio real. No caso de Lung Fibrosis, esse comportamento é consistente com as anomalias já observadas no ImagSTFT, RealSTFT e MFCC, reforçando a hipótese de que a amostra de índice 960 apresenta alguma irregularidade na gravação ou no processamento que a torna pouco representativa da classe. Já a anomalia em Bronchiolitis é nova em relação às transformações anteriores, nas quais essa classe apresentava comportamento regular, o que sugere a possibilidade de um problema específico no cálculo da derivada para essa amostra, e não necessariamente um problema no sinal de áudio original.

## Chroma
A representação Chroma resulta em uma matriz de formato (128, 216). Durante a execução, foi emitido um aviso (UserWarning: Trying to estimate tuning from empty frequency set) referente a uma das amostras processadas, indicando que o algoritmo de estimativa de afinação não encontrou energia espectral suficiente para realizar o cálculo. Esse aviso é consistente com as anomalias já identificadas anteriormente em algumas amostras (como Lung Fibrosis), reforçando a hipótese de que tais gravações possuem baixa energia espectral ou características atípicas que afetam diversas transformações ao longo do notebook.
Em relação ao padrão visual, a maioria das classes (Asthma, Bronchiectasis, Bronchiolitis, COPD, Healthy e Pneumonia) apresenta padrão de bandas verticais irregulares, com alternância de intensidade ao longo do tempo, sem concentração persistente em uma única classe de pitch específica. Lung Fibrosis exibe padrão semelhante, porém com menor densidade e intensidade de variação, consistente com a hipótese de menor energia espectral nessa amostra. URTI se destaca por apresentar, além das bandas verticais, uma faixa horizontal de coloração clara e relativamente constante ao longo de praticamente toda a gravação, sugerindo a presença de um componente tonal predominante e persistente no tempo, possivelmente associado a algum ruído de fundo ou característica específica dessa gravação.